# Merfish merfish affine only

The same pair as `merfish-merfish`, held to an affine. `diffeo_start` is the iteration at which the velocity field is allowed to start moving; setting it past `niter` means it never does, so only the affine part is ever fitted.

Upstream's equivalent is `merfish-merfish-alignment-affine-only`. One call does the alignment: `align_stalign_obs` fits a
diffeomorphism straight between two point clouds, rasterizing both sides itself.

## Inputs

In [ ]:
import anndata as ad, numpy as np, pandas as pd

def read_xy(path, x, y, *, skip_first=False):
    df = pd.read_csv(path)
    xy = np.c_[np.asarray(df[x])[1:] if skip_first else df[x],
               np.asarray(df[y])[1:] if skip_first else df[y]].astype(float)
    return ad.AnnData(X=np.zeros((len(xy), 1)), obsm={'spatial': xy})

# Upstream's starting guess, applied to the coordinates rather than passed as `initial_affine`:
# the fit then starts from the identity, and the initialisation stays visible here instead of
# folded into a matrix convention.
def rotated_onto(query, ref, degrees):
    theta = np.deg2rad(-degrees)
    rotation = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    centred = query.obsm['spatial'] - query.obsm['spatial'].mean(0)
    out = query.copy()
    out.obsm['spatial'] = centred @ rotation.T + ref.obsm['spatial'].mean(0)
    return out

ref = read_xy('merfish_data/datasets_mouse_brain_map_BrainReceptorShowcase_Slice2_Replicate2_cell_metadata_S2R2.csv.gz', 'center_x', 'center_y')
query = read_xy('merfish_data/datasets_mouse_brain_map_BrainReceptorShowcase_Slice2_Replicate3_cell_metadata_S2R3.csv.gz', 'center_x', 'center_y')
print(f'{ref.n_obs} reference cells, {query.n_obs} query cells')

# Upstream starts this pair 45 degrees apart.
query = rotated_onto(query, ref, 45)

## The fit

Upstream's own solver values, and squidpy's defaults for everything else.

In [ ]:
from squidpy.experimental.tl import align_stalign_obs

fit = align_stalign_obs(
    ref, query, spatial_key='spatial',
    dx=15.0, blur=1.5, niter=1000, epV=50, diffeo_start=1001,
)
print(f'{fit.n_iter} iterations, objective '
      f'{float(fit.energies[0]):.0f} -> {float(fit.energies[-1]):.0f}')

## Where the cells land

`transform` evaluates the fitted map at each point, so a cell lands where it lands rather than
at the nearest raster cell.

In [ ]:
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

moved = np.asarray(fit.transform(query.obsm['spatial']))

# Whether the fit worked, in the units of the data rather than of the objective. The energy
# can halve while the sections stay as far apart as they started -- that is what an iteration
# budget that ran out looks like, and it is invisible in the energy alone.
tree = cKDTree(ref.obsm['spatial'])
before = tree.query(query.obsm['spatial'])[0]
after = tree.query(moved)[0]
print(f"centroid offset {np.linalg.norm(query.obsm['spatial'].mean(0) - ref.obsm['spatial'].mean(0)):.0f}"
      f" -> {np.linalg.norm(moved.mean(0) - ref.obsm['spatial'].mean(0)):.0f}")
print(f'distance to the nearest reference cell: median {np.median(before):.0f} -> '
      f'{np.median(after):.0f}, 90th percentile {np.percentile(before, 90):.0f} -> '
      f'{np.percentile(after, 90):.0f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
for a, (pts, title) in zip(ax, [(query.obsm['spatial'], 'before'),
                                (moved, 'after the fit')], strict=True):
    a.scatter(*ref.obsm['spatial'].T, s=0.12, alpha=0.3, label='reference')
    a.scatter(*pts.T, s=0.12, alpha=0.3, label='query')
    a.set_title(title); a.set_aspect('equal'); a.invert_yaxis()
    a.set_xticks([]); a.set_yticks([])
ax[0].legend(markerscale=90, loc='lower left', fontsize=8)

## Which cells the fit could actually use

Two sections rarely cover the same tissue. The solver splits the target into three classes as
it goes -- matching, artifact and background -- and reweights them every iteration, so regions
with no counterpart stop pulling on the deformation. Reading those weights back is what turns
a partial overlap into something visible, rather than a fit that merely looks bad: the
unmatched parts are supposed to be unmatched.

In [ ]:
# The weights come back as the solver's internal density rasters, and `align_stalign_obs`
# deliberately returns no axes for them -- "not a frame any real image lives on". Upstream can
# colour its cells by weight because it rasterizes by hand and keeps the grid; delegating that
# to squidpy trades the per-cell view for not having to rebuild the grid from `dx` and
# `raster_expand`, which would be a copy of internals that is wrong if it is half a pixel out.
for name in ('match_weights', 'artifact_weights', 'background_weights'):
    w = getattr(fit, name)
    print(f'{name}: {None if w is None else np.asarray(w).shape}')

matching = np.asarray(fit.match_weights).squeeze()
print(f'{100 * (matching > 0.5).mean():.0f}% of the target raster is mostly-matching, '
      f'{100 * (np.asarray(fit.background_weights).squeeze() > 0.5).mean():.0f}% mostly-background')

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, (w, title) in zip(ax, [(matching, 'matching weight'),
                              (np.asarray(fit.background_weights).squeeze(), 'background weight')],
                         strict=True):
    im = a.imshow(w, cmap='viridis', vmin=0, vmax=1)
    a.set_title(title); a.set_xticks([]); a.set_yticks([])
    fig.colorbar(im, ax=a, fraction=0.046)

The objective's trace. The mixture E step switches on at iteration 50 and the energy changes
definition there, so only the part after the dashed line is one function.

In [ ]:
MIXTURE_GATE = 50
energies = np.asarray(fit.energies)[: fit.n_iter]
descent = energies[MIXTURE_GATE + 1 :]
tail = descent[-max(len(descent) // 10, 1) :]
print(f'after the gate: {descent[0]:.0f} -> {descent[-1]:.0f}, minimum {descent.min():.0f} '
      f'at iteration {MIXTURE_GATE + 1 + int(descent.argmin())}')
print(f'last tenth: mean {tail.mean():.0f}, spread {np.ptp(tail):.0f} '
      f'({100 * np.ptp(tail) / tail.mean():.1f}% of its mean)')
plt.plot(energies, lw=0.8); plt.axvline(MIXTURE_GATE, color='0.6', ls='--', lw=0.8)
plt.xlabel('iteration'); plt.ylabel('objective'); plt.grid(alpha=0.3)